# P1 · Q14 post-run raw audit

Run this notebook on the server containing **q14-20261013T064940065373Z**. Choose **Run All**. It uses **8 workers**, resumes verified checkpoints automatically, and exports **Q14_Raw_Audit_Review.zip** for review.

It reads all **100,248 recorded trials in 784 committed batches** (1,944 pilot; 98,304 evaluation). It checks source and metadata identity, raw/compact hashes, seeded inputs, frozen Q12 policies, CPU chronology and service budgets, quantum-event timing, retained density matrices and fidelity, and reconstruction of every compact record. The exported evidence includes receipts for every batch and 30 samples selected by fixed labels.

**No simulation, calibration, or manuscript editing occurs.** Q12 and Q13 are not rerun. Audit outputs are separate from the original Q14 results. NumPy is the only nonstandard audit dependency; use your existing experiment Python environment.

The compact results and statistical analysis were reviewed separately. This notebook links the server raw records to those exact compact results; it does not repeat statistical inference. Its record checker is the pinned Q14 checker, separate from the simulator engine, with a new post-run coordinator. This is a recorded-data integrity check, not an independent simulator or physical-hardware validation.

## 1. Settings

The default matches your server layout. Change `PROJECT_DIR` or `RUN_DIR` only if you moved the files. Keep the original run intact. A compact review ZIP alone does not contain the raw files needed here.

In [ ]:
from pathlib import Path
PROJECT_DIR = Path.home() / "P1"
RUN_DIR = PROJECT_DIR / "results" / "q14-20261013T064940065373Z"
WORKERS = 8


## 2. Install the embedded audit and run its self-tests

The notebook contains all required audit source files. It installs no packages and downloads nothing. Existing audit code must match exactly. Self-tests use synthetic fixtures, including deliberately corrupted records; passing them is not a pass for your experimental data.

In [ ]:
import base64, gzip, hashlib, json, os, subprocess, sys
BUNDLE_SHA256 = '4d89f8d8e2867f58a76e972be5a8596e6d8b52f1133fa3b954abe235704b4315'
_EMBEDDED = '...'
blob = base64.b64decode(_EMBEDDED)
if hashlib.sha256(blob).hexdigest() != BUNDLE_SHA256:
    raise RuntimeError("Embedded audit checksum mismatch")
sources = json.loads(gzip.decompress(blob))
PROJECT_DIR = PROJECT_DIR.expanduser().resolve()
RUN_DIR = RUN_DIR.expanduser().resolve()
CODE_DIR = PROJECT_DIR / "audit_tools" / ("q14-raw-" + BUNDLE_SHA256[:12])
AUDIT_DIR = PROJECT_DIR / "audits" / (RUN_DIR.name + "-raw-" + BUNDLE_SHA256[:12])
if not RUN_DIR.is_dir():
    raise FileNotFoundError("Original Q14 run not found: " + str(RUN_DIR))
if type(WORKERS) is not int or not 1 <= WORKERS <= 64:
    raise ValueError("WORKERS must be an integer from 1 to 64")
for destination in (CODE_DIR.resolve(), AUDIT_DIR.resolve()):
    if destination == RUN_DIR or RUN_DIR in destination.parents or destination in RUN_DIR.parents:
        raise ValueError("Audit output and code must be separate from the original run")
CODE_DIR.mkdir(parents=True, exist_ok=True)
for name, content in sources.items():
    if Path(name).name != name:
        raise RuntimeError("Invalid embedded filename")
    path = CODE_DIR / name
    data = content.encode()
    if path.exists():
        if path.is_symlink() or path.read_bytes() != data:
            raise RuntimeError("Existing audit source differs: " + str(path))
    else:
        with path.open("xb") as f:
            f.write(data)
AUDIT_ENV = dict(os.environ, PYTHONDONTWRITEBYTECODE="1",
                 OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
result = subprocess.run([sys.executable, "-u", str(CODE_DIR / "q14_raw_tests.py")],
                        env=AUDIT_ENV, capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("Audit self-tests failed; experimental data have not been scanned")
print("Run:", RUN_DIR)
print("Audit output:", AUDIT_DIR)


## 3. Audit all retained records

Progress is printed at least every 15 seconds while batches run. Runtime depends on disk speed. Leave this cell running until it finishes. If interrupted, stop the old process before rerunning; the audit also enforces a lock. Resume rechecks original file hashes before reusing a completed audit checkpoint.

A failure produces diagnostics rather than silently skipping a record. Do not edit the original results to make a check pass. Share the exported audit ZIP if any check fails.

In [ ]:
from datetime import datetime, timezone
import signal, time
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = AUDIT_DIR / ("launcher-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".log")
command = [sys.executable, "-u", str(CODE_DIR / "q14_raw_audit.py"),
           "--run-dir", str(RUN_DIR), "--audit-dir", str(AUDIT_DIR),
           "--workers", str(WORKERS)]
with LOG_PATH.open("w") as log:
    process = subprocess.Popen(command, env=AUDIT_ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1,
                               start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()
    except KeyboardInterrupt:
        os.killpg(process.pid, signal.SIGINT)
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
        print("Interrupted. Saved checkpoints remain. Rerun this cell to resume.")
        raise
if return_code != 0:
    print("Audit did not pass. Run the export cell below and share the ZIP.")
else:
    report = json.loads((AUDIT_DIR / "audit_report.json").read_text())
    print("Verified:", report["audited_trials"], "records;", report["committed_batches"], "batches.")


## 4. Export and return the review ZIP

This refreshes the archive after the launcher log is closed. It includes the report, audit source, receipts and fixed samples; the full simulation traces stay on the server. Run this cell after a failure too. The printed absolute path is authoritative if Jupyter cannot display a download link.

**Send `Q14_Raw_Audit_Review.zip` back for review.** A PASS supports the integrity of the retained records; it does not enlarge the scientific claims.

In [ ]:
from IPython.display import FileLink, display
result = subprocess.run([sys.executable, "-u", str(CODE_DIR / "q14_raw_audit.py"),
                         "--run-dir", str(RUN_DIR), "--audit-dir", str(AUDIT_DIR),
                         "--export-only"], env=AUDIT_ENV, capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("Audit export failed")
ZIP_PATH = AUDIT_DIR / "Q14_Raw_Audit_Review.zip"
print("Return this file:", ZIP_PATH)
print("Size:", round(ZIP_PATH.stat().st_size / 1024**2, 2), "MiB")
# A relative link works when the notebook server serves this directory tree.
try:
    display(FileLink(str(ZIP_PATH.relative_to(Path.cwd()))))
except ValueError:
    print("Download the ZIP through your server's file browser using the path above.")
